# 01 — Analisi esplorativa (EDA)

Obiettivi della settimana 1:
1. capire struttura e qualità dei dati;
2. misurare lo sbilanciamento delle classi;
3. guardare come il tasso di default varia tra i **gruppi sensibili** (sesso, età, lavoratore straniero), base del fairness audit della settimana 4.

Prima di eseguire: `python -m credit_xai.data` dalla cartella del progetto.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from credit_xai import config
from credit_xai.data import load_processed

sns.set_theme(style="whitegrid")
df = load_processed()
df.shape

## 1. Struttura e qualità

In [ ]:
df.head()

In [ ]:
info = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "mancanti": df.isna().sum(),
    "valori_unici": df.nunique(),
})
info

In [ ]:
df[config.NUMERIC_FEATURES].describe().T.round(1)

## 2. Target: quanto è sbilanciato?

In [ ]:
rate = df[config.TARGET].mean()
print(f"Tasso di default: {rate:.1%}  ({df[config.TARGET].sum()} su {len(df)})")
# Con classi sbilanciate l'accuracy inganna: useremo AUC, F1 e calibrazione.

## 3. Distribuzioni numeriche per classe

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.flat, ["duration_months", "credit_amount", "age", "installment_rate"]):
    sns.histplot(data=df, x=col, hue=config.TARGET, stat="density",
                 common_norm=False, element="step", ax=ax)
    ax.set_title(col)
plt.tight_layout()

## 4. Tasso di default per variabile categorica

In [ ]:
def default_rate(col):
    return (df.groupby(col)[config.TARGET]
              .agg(tasso_default="mean", n="size")
              .sort_values("tasso_default", ascending=False))

for col in ["checking_status", "credit_history", "savings", "purpose"]:
    display(default_rate(col).style.format({"tasso_default": "{:.1%}"}))

## 5. Gruppi sensibili

Questi attributi **non** entreranno nel modello, ma ci servono per controllare che le decisioni non penalizzino un gruppo. Attenzione ai proxy: altre feature (ad es. `personal_status`, `job`) possono essere correlate al sesso o all'età.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, col in zip(axes, config.SENSITIVE_ATTRIBUTES):
    rates = df.groupby(col)[config.TARGET].mean().sort_index()
    rates.plot.bar(ax=ax, color="#4C72B0")
    ax.axhline(df[config.TARGET].mean(), ls="--", c="grey", label="media")
    ax.set_title(f"Tasso di default per {col}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
axes[0].legend()
plt.tight_layout()

In [ ]:
# Possibili proxy: quanto ogni feature categorica 'rivela' il sesso?
pd.crosstab(df["personal_status"], df["sex"])

## 6. Correlazioni tra feature numeriche

In [ ]:
corr = df[config.NUMERIC_FEATURES + [config.TARGET]].corr()
plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlazioni")

## 7. Note per le prossime settimane

Annota qui cosa hai osservato, ti servirà per README e model card:

- Tasso di default e sbilanciamento: …
- Feature più associate al default: …
- Differenze tra gruppi sensibili: …
- Possibili proxy di attributi protetti: …
- Limiti del dataset (anno di raccolta, valuta in DM, dimensione ridotta): …